# CLUSTERING: Selection of RRLs with G > 18 mag from 'vari_classifier_result.csv' catalogue

## Libraries

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from astropy.io import ascii, fits
from astropy.visualization import ZScaleInterval, ImageNormalize
from astropy.io.votable import parse, parse_single_table
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table, join

from shapely.geometry import Point, Polygon

import csv

import pandas as pd

from scipy.stats import norm, lognorm, expon, gamma

from statistics import mode

import seaborn as sns

from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset


import warnings
warnings.filterwarnings('ignore')

## Wesenheit Relation for RR Lyrae ([Garofalo et al. 2022](https://doi.org/10.1093/mnras/stac735))

In [2]:
## Garofalo et al. 2022, pag. 12-13

def wesen(g, bp, rp) :
    w = g - 1.922 * (bp - rp)
    
    return w

## Data Download

In [3]:
data = pd.read_csv('Data/vari_classifier_result.csv', header=0)

## Selection of RRLs with G > 18 mag from 'vari_classifier_result.csv' catalogue

In [4]:
print('Sources: ' + str(len(data)))

data = data.drop(columns = ['ra_error', 'dec_error', 'parallax', 'parallax_error', 'pm', 'phot_g_mean_flux', 'phot_g_mean_flux_error', 
                            'phot_bp_mean_flux', 'phot_bp_mean_flux_error', 'phot_rp_mean_flux', 'phot_rp_mean_flux_error'])


## Mask for Sources with non-Nan pmra, pmdec, phot_g bp rp, int_g bp rp

mask_initial = ((np.isnan(data['pmra']) == False) & (np.isnan(data['pmdec']) == False) &
               (np.isnan(data['phot_g_mean_mag']) == False) & (np.isnan(data['phot_bp_mean_mag']) == False) &
               (np.isnan(data['phot_rp_mean_mag']) == False))

data = data[mask_initial]

print('Sources: ' + str(len(data)))


# G > 18 mag

mask_18mag = (data['phot_g_mean_mag'] > 18.0)

data = data[mask_18mag]

print('Sources with G > 18 mag: ' + str(len(data)))

Sources: 297778
Sources: 277196
Sources with G > 18 mag: 155127


## Computation of Wesenheit Magnitude for each source

In [5]:
w_mag = wesen(data['phot_g_mean_mag'], data['phot_bp_mean_mag'], data['phot_rp_mean_mag'])

data.insert(len(data.columns), 'w_mag', w_mag)

print(data)

                  source_id          ra        dec      pmra  pmra_error  \
1       5902578129041728384  230.641025 -48.902621 -7.448976    0.360380   
3       6070183489684676224  202.933991 -50.516881 -6.446995    0.410479   
5       5937150622937629312  251.361867 -50.602962 -6.565843    2.541872   
7       6030044168448081664  255.127449 -29.135600 -4.077028    1.732087   
18      6027876103330357376  251.856259 -31.462487  0.447601    0.593153   
...                     ...         ...        ...       ...         ...   
297770  6029924081179305088  256.639244 -28.969524 -7.579734    0.674993   
297771  6029972768966134144  256.182882 -28.845451 -0.141074    0.844168   
297772  6030010977026362880  256.607556 -28.388314 -5.349512    0.946197   
297774  4124054436882600064  265.559059 -16.608629 -0.586920    1.819907   
297777  5917294508087466624  256.341820 -56.328720 -3.727112    0.958507   

           pmdec  pmdec_error  phot_g_mean_mag  phot_bp_mean_mag  \
1      -3.928572   

## File saving

In [6]:
data.to_csv('Data/vari_class_18mag.csv', index=False)